# Pipeline ETL con la API oficial del BCRA y Delta Lake

**Autor:** Renzo Gutierrez  
**Fuente:** API oficial de Estadísticas Cambiarias del Banco Central de la República Argentina (BCRA)

## Objetivo

Construir un pipeline ETL reproducible que:

1. Extraiga el maestro de divisas y la serie histórica del dólar estadounidense desde la API oficial del BCRA.
2. Transforme las respuestas JSON anidadas en tablas analíticas.
3. Almacene los datos en Delta Lake sobre MinIO.
4. Implemente una carga completa para datos maestros y una carga incremental para la serie temporal.
5. Incorpore paginación, reintentos, controles de calidad, idempotencia y mantenimiento con `VACUUM`.

Esta notebook reemplaza la API comunitaria `api.estadisticasbcra.com`, actualmente discontinuada, por la fuente oficial del BCRA. Está escrita con explicaciones y comentarios para que también funcione como pieza de portfolio sobre conocimientos de ETL y arquitectura de datos.


## 1. Diseño y decisiones técnicas

### Endpoints seleccionados

| Dataset | Endpoint | Tipo de extracción | Estrategia de carga |
|---|---|---|---|
| Maestro de divisas | `/Maestros/Divisas` | Completa | `overwrite` |
| Cotización USD | `/Cotizaciones/USD` | Incremental por rango de fechas | `MERGE` por fecha |

La antigua API incluía un endpoint `milestones`, pero la API oficial no ofrece un equivalente. Para conservar el objetivo didáctico de trabajar con datos estáticos y datos incrementales, se utiliza el maestro de divisas como dataset de referencia.

### ¿Por qué estas estrategias?

- **Maestro de divisas:** es pequeño y puede cambiar en altas, bajas o denominaciones. Reemplazar el snapshot completo simplifica su mantenimiento.
- **Cotización USD:** es una serie temporal. Después de la primera carga sólo se solicitan fechas posteriores al máximo almacenado, reduciendo transferencia, tiempo y procesamiento.
- **Paginación:** la API limita la cantidad de registros por respuesta; se utilizan `limit` y `offset` hasta recuperar todo el rango.
- **Delta Lake:** aporta transacciones ACID, control de esquema, historial y operaciones `MERGE` sobre almacenamiento de objetos.
- **MinIO:** ofrece una interfaz compatible con S3 para desacoplar cómputo y almacenamiento.
- **VACUUM:** elimina archivos de datos obsoletos luego del período de retención. No elimina el historial activo del `_delta_log`, necesario para mantener la consistencia de la tabla.

### Flujo

`API BCRA → extracción HTTP → transformación con pandas → validaciones → Delta Lake en MinIO → control de calidad y visualización`

Documentación oficial:  
- https://www.bcra.gob.ar/documentacion-apis/?fileName=estadisticascambiarias-v1  
- https://www.bcra.gob.ar/archivos/Catalogo/Content/files/pdf/estadisticascambiarias-v1.pdf


## 2. Dependencias

Se instalan únicamente las librerías necesarias para consumir la API, transformar datos, escribir tablas Delta y visualizar el resultado.


In [ ]:
%pip install -q requests pandas pyarrow deltalake matplotlib

In [ ]:
import os
from datetime import date, datetime, timedelta
from getpass import getpass
from typing import Any
from zoneinfo import ZoneInfo

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from deltalake import DeltaTable, write_deltalake
from deltalake.exceptions import TableNotFoundError

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 3. Configuración

La API del BCRA es pública y **no requiere token**. Las credenciales siguientes corresponden únicamente a MinIO.

La fecha máxima de consulta se calcula con la zona horaria de Buenos Aires. Esto evita enviar una fecha futura cuando la notebook corre en un servidor configurado en UTC.

Para evitar publicar secretos en GitHub o LinkedIn, la notebook busca variables de entorno y, si no existen, solicita los valores de forma interactiva. `getpass` evita que las credenciales se muestren en pantalla.

Variables opcionales:

- `MINIO_ENDPOINT`
- `MINIO_ACCESS_KEY`
- `MINIO_SECRET_KEY`
- `MINIO_BUCKET`


In [ ]:
# Configuración de la API oficial del BCRA.
API_BASE_URL = "https://api.bcra.gob.ar/estadisticascambiarias/v1.0"
CURRENCY_CODE = "USD"

# Fecha desde la cual se intentará construir el histórico en la primera ejecución.
# Si la API no posee datos para los primeros días, simplemente devolverá los disponibles.
HISTORICAL_START_DATE = date(2000, 1, 1)

# El manual establece paginación mediante limit y offset.
# Se utiliza 999 para permanecer dentro del máximo documentado.
PAGE_SIZE = 999
REQUEST_TIMEOUT_SECONDS = 30
VACUUM_RETENTION_HOURS = 168  # 7 días
ARGENTINA_TIME_ZONE = ZoneInfo("America/Argentina/Buenos_Aires")


def argentina_today() -> date:
    """Devuelve la fecha vigente en Argentina, independientemente del servidor."""
    return datetime.now(ARGENTINA_TIME_ZONE).date()


def get_setting(name: str, prompt: str, secret: bool = False) -> str:
    """Obtiene una configuración desde el entorno o la solicita al usuario."""
    value = os.getenv(name)
    if value:
        return value
    return getpass(prompt) if secret else input(prompt).strip()


# Las credenciales no quedan almacenadas dentro de la notebook.
MINIO_ENDPOINT = get_setting("MINIO_ENDPOINT", "Endpoint de MinIO: ")
MINIO_ACCESS_KEY = get_setting("MINIO_ACCESS_KEY", "Access key de MinIO: ", secret=True)
MINIO_SECRET_KEY = get_setting("MINIO_SECRET_KEY", "Secret key de MinIO: ", secret=True)
MINIO_BUCKET = get_setting("MINIO_BUCKET", "Bucket de MinIO: ")

storage_options = {
    "AWS_ENDPOINT_URL": MINIO_ENDPOINT,
    "AWS_ACCESS_KEY_ID": MINIO_ACCESS_KEY,
    "AWS_SECRET_ACCESS_KEY": MINIO_SECRET_KEY,
    "AWS_ALLOW_HTTP": "true",
    "aws_conditional_put": "etag",
    "AWS_S3_ALLOW_UNSAFE_RENAME": "true",
}

# Las rutas identifican explícitamente la fuente de los datos.
BRONZE_BASE_PATH = (
    f"s3://{MINIO_BUCKET}/lakehouse/bronze/"
    "bcra_estadisticas_cambiarias_api"
)
CURRENCIES_PATH = f"{BRONZE_BASE_PATH}/currencies"
USD_QUOTES_PATH = f"{BRONZE_BASE_PATH}/usd_quotes"

print(f"Ruta base configurada: {BRONZE_BASE_PATH}")

## 4. Cliente HTTP resiliente

Las APIs pueden responder temporalmente con errores o límites de tráfico. La sesión implementa reintentos exponenciales para códigos `429` y errores de servidor, y centraliza la validación de las respuestas.


In [ ]:
def build_http_session() -> requests.Session:
    """Crea una sesión HTTP con reintentos para fallos transitorios."""
    retry_strategy = Retry(
        total=3,
        backoff_factor=1,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
    )

    adapter = HTTPAdapter(max_retries=retry_strategy)
    session = requests.Session()
    session.mount("https://", adapter)
    return session


HTTP_SESSION = build_http_session()


def get_json(endpoint: str, params: dict[str, Any] | None = None) -> dict:
    """Ejecuta un GET y valida tanto HTTP como el status informado por la API."""
    url = f"{API_BASE_URL}/{endpoint.lstrip('/')}"

    response = HTTP_SESSION.get(
        url,
        params=params,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()
    if payload.get("status") != 200:
        raise RuntimeError(
            f"La API devolvió status={payload.get('status')}: "
            f"{payload.get('errorMessages', payload)}"
        )

    return payload

## 5. Extract

### 5.1 Extracción completa del maestro de divisas

El maestro es pequeño, por lo que se obtiene completo en cada ejecución.


In [ ]:
def extract_currencies() -> list[dict]:
    """Extrae el catálogo completo de monedas disponibles."""
    payload = get_json("Maestros/Divisas")
    return payload["results"]

### 5.2 Extracción incremental y paginada de USD

El endpoint permite filtrar con `fechadesde` y `fechahasta`. La función recorre todas las páginas y detiene el ciclo cuando recibe menos registros que el tamaño solicitado.

En las ejecuciones posteriores, la fecha inicial se calcula a partir del máximo `date` existente en Delta Lake. Así se evita descargar nuevamente todo el histórico.


In [ ]:
def extract_usd_quotes(
    date_from: date,
    date_to: date,
    page_size: int = PAGE_SIZE,
) -> list[dict]:
    """Extrae la cotización USD para un rango, recorriendo todas las páginas."""
    if date_from > date_to:
        return []

    all_records = []
    offset = 0
    total_count = None

    while True:
        params = {
            "fechadesde": date_from.isoformat(),
            "fechahasta": date_to.isoformat(),
            "limit": page_size,
            "offset": offset,
        }

        payload = get_json(f"Cotizaciones/{CURRENCY_CODE}", params=params)
        page = payload.get("results", [])
        resultset = payload.get("metadata", {}).get("resultset", {})
        total_count = int(resultset.get("count", 0))

        # Acumulamos la página actual antes de decidir si corresponde continuar.
        all_records.extend(page)
        print(
            f"Página offset={offset}: {len(page)} registros "
            f"(count informado={total_count})"
        )

        # La API puede devolver menos filas que el límite aun cuando queden
        # páginas pendientes, por ejemplo por fines de semana y feriados.
        # Por eso no se utiliza len(page) < page_size como condición final.
        if not page:
            break

        # offset representa posiciones de la API, no filas efectivamente
        # recibidas. Debe avanzar por el límite solicitado.
        offset += page_size
        if total_count and offset >= total_count:
            break

    return all_records

## 6. Transform

La API devuelve nombres en español y una estructura anidada para las cotizaciones. La transformación:

- normaliza nombres de columnas;
- convierte fechas y valores a tipos explícitos;
- incorpora la fecha de extracción para trazabilidad;
- elimina duplicados por clave natural;
- ordena el resultado para facilitar validaciones.


In [ ]:
def transform_currencies(data: list[dict]) -> pd.DataFrame:
    """Convierte el maestro de divisas en una tabla normalizada."""
    df = pd.DataFrame(data).rename(
        columns={
            "codigo": "currency_code",
            "denominacion": "currency_name",
        }
    )

    df["currency_code"] = df["currency_code"].astype("string").str.strip()
    df["currency_name"] = df["currency_name"].astype("string").str.strip()
    df["extracted_at_utc"] = pd.Timestamp.now(tz="UTC")

    return (
        df[["currency_code", "currency_name", "extracted_at_utc"]]
        .drop_duplicates(subset=["currency_code"], keep="last")
        .sort_values("currency_code")
        .reset_index(drop=True)
    )


def transform_usd_quotes(data: list[dict]) -> pd.DataFrame:
    """Aplana la respuesta anidada y genera una fila por fecha."""
    rows = []

    for record in data:
        # Para USD debería existir un único elemento dentro de detalle.
        # Se recorre la lista para no depender de una posición fija.
        for detail in record.get("detalle", []):
            rows.append(
                {
                    "date": record.get("fecha"),
                    "currency_code": detail.get("codigoMoneda"),
                    "currency_name": detail.get("descripcion"),
                    "exchange_rate_ars": detail.get("tipoCotizacion"),
                    "pass_through_rate": detail.get("tipoPase"),
                }
            )

    columns = [
        "date",
        "currency_code",
        "currency_name",
        "exchange_rate_ars",
        "pass_through_rate",
        "extracted_at_utc",
    ]

    if not rows:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(rows)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["exchange_rate_ars"] = pd.to_numeric(
        df["exchange_rate_ars"], errors="coerce"
    )
    df["pass_through_rate"] = pd.to_numeric(
        df["pass_through_rate"], errors="coerce"
    )
    df["extracted_at_utc"] = pd.Timestamp.now(tz="UTC")

    # Una fecha es la clave natural elegida para la serie USD.
    return (
        df[columns]
        .dropna(subset=["date", "exchange_rate_ars"])
        .drop_duplicates(subset=["date"], keep="last")
        .sort_values("date")
        .reset_index(drop=True)
    )

## 7. Controles de calidad

Antes de escribir se comprueban columnas, nulos, duplicados y valores positivos. Si una regla crítica falla, el pipeline se detiene para evitar cargar datos inválidos.


In [ ]:
def validate_currencies(df: pd.DataFrame) -> None:
    """Valida el esquema mínimo y la unicidad del maestro."""
    required = {"currency_code", "currency_name", "extracted_at_utc"}
    missing = required.difference(df.columns)

    if missing:
        raise ValueError(f"Faltan columnas en currencies: {sorted(missing)}")
    if df.empty:
        raise ValueError("El maestro de divisas está vacío.")
    if df["currency_code"].isna().any():
        raise ValueError("Existen códigos de moneda nulos.")
    if df["currency_code"].duplicated().any():
        raise ValueError("Existen códigos de moneda duplicados.")


def validate_usd_quotes(df: pd.DataFrame) -> None:
    """Valida las reglas de calidad de la serie temporal USD."""
    required = {"date", "currency_code", "exchange_rate_ars"}
    missing = required.difference(df.columns)

    if missing:
        raise ValueError(f"Faltan columnas en usd_quotes: {sorted(missing)}")
    if df.empty:
        raise ValueError("La extracción de USD no devolvió registros.")
    if df["date"].isna().any():
        raise ValueError("Existen fechas nulas o inválidas.")
    if df["date"].duplicated().any():
        raise ValueError("Existen fechas duplicadas.")
    if (df["exchange_rate_ars"] <= 0).any():
        raise ValueError("Existen cotizaciones nulas o negativas.")

## 8. Load: Delta Lake sobre MinIO

Se implementan dos patrones:

- `save_snapshot_as_delta`: reemplaza el maestro completo.
- `upsert_as_delta`: actualiza coincidencias e inserta fechas nuevas mediante `MERGE`.

La conversión a PyArrow usa `preserve_index=False` para evitar guardar el índice de pandas como una columna adicional.


In [ ]:
def to_arrow(df: pd.DataFrame) -> pa.Table:
    """Convierte un DataFrame a Arrow sin materializar el índice."""
    return pa.Table.from_pandas(df, preserve_index=False)


def save_snapshot_as_delta(
    df: pd.DataFrame,
    data_path: str,
    storage_options: dict,
) -> None:
    """Guarda un snapshot completo usando overwrite."""
    write_deltalake(
        data_path,
        to_arrow(df),
        mode="overwrite",
        storage_options=storage_options,
    )


def upsert_as_delta(
    df: pd.DataFrame,
    data_path: str,
    predicate: str,
    storage_options: dict,
) -> None:
    """Ejecuta un MERGE idempotente; crea la tabla si todavía no existe."""
    try:
        delta_table = DeltaTable(data_path, storage_options=storage_options)

        (
            delta_table.merge(
                source=to_arrow(df),
                source_alias="source",
                target_alias="target",
                predicate=predicate,
            )
            .when_matched_update_all()
            .when_not_matched_insert_all()
            .execute()
        )
    except TableNotFoundError:
        write_deltalake(
            data_path,
            to_arrow(df),
            mode="overwrite",
            storage_options=storage_options,
        )


def vacuum_delta_table(
    data_path: str,
    storage_options: dict,
    retention_hours: int = VACUUM_RETENTION_HOURS,
) -> list[str]:
    """Elimina archivos obsoletos respetando la retención de seguridad."""
    delta_table = DeltaTable(data_path, storage_options=storage_options)
    deleted_files = delta_table.vacuum(
        retention_hours=retention_hours,
        dry_run=False,
        enforce_retention_duration=True,
    )

    print(f"VACUUM {data_path}: {len(deleted_files)} archivos eliminados")
    return deleted_files

## 9. Watermark incremental

El *watermark* es la última fecha cargada correctamente. La próxima extracción comienza al día siguiente. Además de controlar el máximo, se inspecciona la fecha mínima: si la tabla comienza mucho después de `HISTORICAL_START_DATE`, el pipeline ejecuta un backfill para reparar el tramo histórico faltante.


In [ ]:
def get_stored_date_bounds(
    data_path: str,
    storage_options: dict,
) -> tuple[date | None, date | None]:
    """Devuelve las fechas mínima y máxima almacenadas, si existen."""
    try:
        delta_table = DeltaTable(data_path, storage_options=storage_options)
        dates = pd.to_datetime(
            delta_table.to_pandas(columns=["date"])["date"],
            errors="coerce",
        ).dropna()

        if dates.empty:
            return None, None

        return dates.min().date(), dates.max().date()
    except TableNotFoundError:
        return None, None


def get_next_extraction_date(
    data_path: str,
    storage_options: dict,
    first_date: date = HISTORICAL_START_DATE,
) -> date:
    """Calcula la primera fecha que todavía no fue cargada."""
    _, last_loaded_date = get_stored_date_bounds(
        data_path, storage_options
    )
    if last_loaded_date is None:
        return first_date
    return last_loaded_date + timedelta(days=1)

## 10. Orquestación del pipeline

La función `run_pipeline` integra todas las etapas. El orden es importante: primero se extrae y valida; sólo después se modifica el almacenamiento.


In [ ]:
def extract_validate_upsert_usd(
    date_from: date,
    date_to: date,
    extraction_label: str,
) -> int:
    """Extrae un rango, lo valida y lo integra de forma idempotente."""
    if date_from > date_to:
        return 0

    print(f"{extraction_label}: USD desde {date_from} hasta {date_to}")
    usd_raw = extract_usd_quotes(date_from, date_to)
    usd_df = transform_usd_quotes(usd_raw)

    if usd_df.empty:
        print(f"{extraction_label}: el rango no contiene cotizaciones.")
        return 0

    validate_usd_quotes(usd_df)
    upsert_as_delta(
        usd_df,
        USD_QUOTES_PATH,
        predicate="target.date = source.date",
        storage_options=storage_options,
    )
    return len(usd_df)


def run_pipeline() -> dict[str, Any]:
    """Ejecuta el pipeline ETL completo y devuelve métricas de la corrida."""
    run_started_at = pd.Timestamp.now(tz="UTC")

    # ------------------------------------------------------------------
    # 1. MAESTRO DE DIVISAS: extracción completa + overwrite
    # ------------------------------------------------------------------
    currencies_raw = extract_currencies()
    currencies_df = transform_currencies(currencies_raw)
    validate_currencies(currencies_df)

    save_snapshot_as_delta(
        currencies_df,
        CURRENCIES_PATH,
        storage_options,
    )
    vacuum_delta_table(CURRENCIES_PATH, storage_options)

    # ------------------------------------------------------------------
    # 2. USD: reparación histórica + extracción incremental + MERGE
    # ------------------------------------------------------------------
    # La API valida la fecha contra el día vigente en Argentina.
    # Usar date.today() podría adelantar un día si el servidor trabaja en UTC.
    date_to = argentina_today()
    first_loaded_date, _ = get_stored_date_bounds(
        USD_QUOTES_PATH, storage_options
    )

    usd_backfill_rows_loaded = 0
    historical_tolerance = HISTORICAL_START_DATE + timedelta(days=10)

    # Si la tabla existe pero comienza años después del inicio solicitado,
    # se recupera el segmento faltante sin borrar los datos ya almacenados.
    if (
        first_loaded_date is not None
        and first_loaded_date > historical_tolerance
    ):
        backfill_end = first_loaded_date - timedelta(days=1)
        usd_backfill_rows_loaded = extract_validate_upsert_usd(
            HISTORICAL_START_DATE,
            backfill_end,
            extraction_label="Backfill histórico",
        )

    date_from = get_next_extraction_date(
        USD_QUOTES_PATH,
        storage_options,
    )
    usd_incremental_rows_loaded = extract_validate_upsert_usd(
        date_from,
        date_to,
        extraction_label="Carga incremental",
    )
    usd_rows_loaded = (
        usd_backfill_rows_loaded + usd_incremental_rows_loaded
    )

    if usd_rows_loaded > 0:
        vacuum_delta_table(USD_QUOTES_PATH, storage_options)
    else:
        print("La tabla USD ya está actualizada.")

    run_finished_at = pd.Timestamp.now(tz="UTC")

    metrics = {
        "run_started_at": run_started_at,
        "run_finished_at": run_finished_at,
        "duration_seconds": (run_finished_at - run_started_at).total_seconds(),
        "currencies_rows": len(currencies_df),
        "usd_rows_loaded": usd_rows_loaded,
        "usd_backfill_rows_loaded": usd_backfill_rows_loaded,
        "usd_incremental_rows_loaded": usd_incremental_rows_loaded,
        "usd_date_from": date_from,
        "usd_date_to": date_to,
    }

    print("Pipeline finalizado correctamente.")
    return metrics

In [ ]:
pipeline_metrics = run_pipeline()
pd.Series(pipeline_metrics, name="value").to_frame()

## 11. Validación posterior a la carga

La lectura final comprueba el estado realmente persistido, no solamente el DataFrame que existía en memoria antes de escribir.


In [ ]:
currencies_delta = DeltaTable(
    CURRENCIES_PATH,
    storage_options=storage_options,
).to_pandas()

usd_delta = DeltaTable(
    USD_QUOTES_PATH,
    storage_options=storage_options,
).to_pandas()

usd_delta["date"] = pd.to_datetime(usd_delta["date"])
usd_delta = usd_delta.sort_values("date").reset_index(drop=True)

quality_summary = pd.DataFrame(
    {
        "metric": [
            "Divisas almacenadas",
            "Cotizaciones USD almacenadas",
            "Primera fecha USD",
            "Última fecha USD",
            "Fechas USD duplicadas",
            "Cotizaciones USD nulas",
        ],
        "value": [
            len(currencies_delta),
            len(usd_delta),
            usd_delta["date"].min(),
            usd_delta["date"].max(),
            int(usd_delta["date"].duplicated().sum()),
            int(usd_delta["exchange_rate_ars"].isna().sum()),
        ],
    }
)

quality_summary

## 12. Visualización del resultado

Aunque el objetivo central es ETL, una visualización rápida ayuda a detectar saltos, valores faltantes o comportamientos inesperados en la serie cargada.


In [ ]:
plot_data = usd_delta.dropna(subset=["date", "exchange_rate_ars"])

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(
    plot_data["date"],
    plot_data["exchange_rate_ars"],
    color="#2563EB",
    linewidth=1.8,
)

ax.set_title("Evolución de la cotización USD publicada por el BCRA")
ax.set_xlabel("Fecha")
ax.set_ylabel("Cotización en ARS")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 13. Conclusiones y aprendizajes

Este proyecto muestra varios componentes habituales de un pipeline de datos productivo:

- migración desde una fuente discontinuada hacia una API oficial;
- extracción completa e incremental;
- paginación y reintentos ante fallos transitorios;
- transformación de JSON anidado a un modelo tabular;
- controles de calidad antes y después de la escritura;
- cargas idempotentes mediante `MERGE`;
- almacenamiento transaccional con Delta Lake sobre MinIO;
- mantenimiento de archivos con `VACUUM`;
- métricas básicas de ejecución y observabilidad.

### Ideas para una siguiente versión

- agregar tests unitarios con respuestas simuladas;
- ejecutar el pipeline con un scheduler;
- registrar métricas y errores en una tabla de auditoría;
- aplicar particionado cuando el volumen justifique su costo;
- crear una capa Silver con reglas de negocio y una capa Gold para visualización.

Esta notebook puede acompañar un post de LinkedIn explicando no sólo el resultado, sino también las decisiones de diseño: por qué se eligió una carga incremental, cómo se garantiza idempotencia y qué problema resuelve Delta Lake en un data lake.
